# Exercício: frete grátis vale a pena?

**Disciplina 4 — Decisão com Dados · Intervalos de Confiança**

## O caso

Marketing da Olist quer lançar a promoção **"frete grátis em compras a partir de R$ 99"**. Antes de aprovar, Financeiro quer saber:

1. Quanto custa, em média, cada frete que a empresa passaria a pagar?
2. Quantos pedidos ficariam elegíveis para a promoção?
3. Quanto isso custa a cada 1.000 pedidos, em cada região?
4. Em que região a promoção cabe no orçamento?

**Orçamento aprovado por Financeiro:** até **R$ 15.000 a cada 1.000 pedidos** (R$ 15 por pedido, em média).

**Regra de decisão:** como na aula, vamos decidir pelo **cenário pessimista**, usando os limites dos ICs, e não só pela estimativa.

## Como fazer

- Cada tarefa traz uma **dica** de qual função usar. Os ICs devem ser calculados com as bibliotecas (`scipy.stats` e `statsmodels`).
- Pode escrever o código direto ou montar um prompt para a IA. Se usar IA, **leia o código** antes de rodar e confira se ele faz o que a tarefa pede.
- Depois de cada resultado, escreva **uma ou duas frases** de leitura: o que o número diz para o negócio.
- Base: pedidos **entregues**, comprados entre **janeiro e agosto de 2018** (o mesmo recorte da aula).

## 0. Carregando os dados (pronto, só rodar)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

url = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"

pedidos = pd.read_csv(url + "olist_orders_dataset.csv")
itens = pd.read_csv(url + "olist_order_items_dataset.csv")
clientes = pd.read_csv(url + "olist_customers_dataset.csv")

pedidos["order_purchase_timestamp"] = pd.to_datetime(pedidos["order_purchase_timestamp"])

# região de cada UF
uf_para_regiao = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

print("pedidos:", pedidos.shape, "| itens:", itens.shape, "| clientes:", clientes.shape)

## Tarefa 1. Montar a base: um pedido por linha

A tabela `itens` tem **um item por linha**: um pedido com 3 produtos aparece 3 vezes. Para a promoção, o que importa é o **pedido inteiro**.

1. Filtre `pedidos`: `order_status == "delivered"` e compra entre `"2018-01-01"` (inclusive) e `"2018-09-01"` (exclusive).
2. Em `itens`, agrupe por `order_id` e calcule `valor_produtos` (soma de `price`) e `frete` (soma de `freight_value`).
3. Junte as três tabelas numa tabela `base`: pedidos filtrados + valores por pedido (pelo `order_id`) + `customer_state` dos clientes (pelo `customer_id`).
4. Crie as colunas:
   - `regiao`: use o dicionário `uf_para_regiao`
   - `elegivel`: True se `valor_produtos >= 99`
5. Mostre quantos pedidos ficaram e o `describe` de `valor_produtos` e `frete`.

**Dica:** `groupby("order_id").agg(...)`, `merge(..., on=...)`, `.map(uf_para_regiao)`.

**Para conferir:** devem ficar **52.783 pedidos**.

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 2. Quanto custa um frete? (IC para média)

1. Calcule o **frete médio por pedido** no Brasil, com IC de 95%.
2. Faça o mesmo **por região**: uma tabela com `regiao`, `n`, `frete_medio`, `li` e `ls`, ordenada pelo frete médio.

**Dica:** `stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(dados))`. Use um loop pelas regiões e guarde os resultados numa lista de dicionários.

**Pergunta:** em que região o frete é mais caro? A diferença para o Sudeste é grande?

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 3. Quantos pedidos ficam elegíveis? (IC para proporção)

1. Calcule a **proporção de pedidos elegíveis** (valor ≥ R$ 99) no Brasil, com IC de Wilson.
2. Faça o mesmo por região: `regiao`, `n`, `x` (elegíveis), `prop_%`, `li_%`, `ls_%`.

**Dica:** `proportion_confint(x, n, alpha=0.05, method="wilson")`, onde `x` é o número de elegíveis.

**Pergunta:** a proporção de elegíveis muda muito entre regiões?

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 4. Quanto custa a cada 1.000 pedidos? E cabe no orçamento?

A empresa só paga o frete dos pedidos **elegíveis**. Então, a cada 1.000 pedidos:

**custo = 1.000 × (proporção de elegíveis) × (frete médio dos elegíveis)**

Atenção: o frete que entra na conta é o **frete médio dos pedidos elegíveis**, não o de todos os pedidos.

Para cada região:
1. Calcule a proporção de elegíveis e seu IC de Wilson.
2. Calcule o frete médio **só dos elegíveis** e seu IC (t).
3. Calcule:
   - `custo_estimado` = 1.000 × proporção × frete médio dos elegíveis
   - `custo_pessimista` = 1.000 × **limite superior** da proporção × **limite superior** do frete
4. Crie a coluna `aprova`: True se o custo pessimista for ≤ R$ 15.000.

**Pergunta:** em quais regiões a promoção cabe no orçamento, mesmo no pior cenário?

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 5. A diferença entre regiões é real? (IC da diferença de médias)

Marketing argumenta: "a diferença de frete entre Nordeste e Sudeste pode ser ruído". Responda com dados.

1. Calcule a diferença de frete médio **Nordeste − Sudeste**, com IC de 95%.
2. O IC contém zero?

**Dica:**

```python
from statsmodels.stats.weightstats import DescrStatsW, CompareMeans
comparacao = CompareMeans(DescrStatsW(nordeste), DescrStatsW(sudeste))
li, ls = comparacao.tconfint_diff(alpha=0.05, usevar="unequal")
```

`usevar="unequal"` não supõe que as duas regiões tenham a mesma variância.

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 6. Onde o frete mais dói? (Wald × Wilson)

Um argumento a favor da promoção: em alguns pedidos, **o frete é mais caro que o próprio produto**, e isso afasta o cliente.

1. Crie a coluna `frete_caro`: True se `frete > valor_produtos`.
2. Calcule a proporção de pedidos com `frete_caro` no Brasil, com IC de Wilson.
3. Para as **5 UFs com menos pedidos**, calcule a proporção com dois ICs: Wald (`method="normal"`) e Wilson (`method="wilson"`).

**Pergunta:** o que acontece com o IC de Wald em Roraima e no Amapá? Qual dos dois você usaria para reportar à diretoria?

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 7. Quantos pedidos para um piloto? (tamanho de amostra)

Financeiro topa um **piloto** da promoção no Sudeste e quer estimar, durante o piloto:

- o **frete médio dos pedidos elegíveis**, com margem de **±R$ 1**;
- a **proporção de elegíveis**, com margem de **±2 pontos percentuais**.

1. Para o frete, use o desvio-padrão do frete dos elegíveis do Sudeste como palpite de σ e calcule n = (z · σ / E)², com z = 1,96 e E = 1. Arredonde para cima.
2. Para a proporção, use a proporção de elegíveis do Sudeste como palpite e calcule o n com `samplesize_confint_proportion`.
3. Quantos pedidos o piloto precisa, considerando as duas metas?

**Dica:**

```python
from statsmodels.stats.proportion import samplesize_confint_proportion
n = samplesize_confint_proportion(proportion=p, half_length=0.02, alpha=0.05)
```

In [ ]:
# seu código aqui


**Sua leitura:** 

## Tarefa 8. A recomendação

Escreva, em 3 ou 4 frases, a recomendação para Financeiro e Marketing. Use os números e os ICs que você calculou. Responda:

- Lançar a promoção? Onde?
- Por que não nas outras regiões?
- Qual é o próximo passo?

**Sua leitura:** 

---
## Desafios (opcional)

1. **Valor mínimo por região.** Para o Nordeste, qual valor mínimo de compra (R$ 99, 150, 200, 250...) faria a promoção caber no orçamento no cenário pessimista?
2. **Limitação.** A conta supõe que o comportamento do cliente não muda com a promoção. O que provavelmente vai acontecer com a proporção de elegíveis quando a promoção existir? Isso aumenta ou diminui o custo?